In [1]:
import os
import json
import time
import minsearch
import pandas as pd

from tqdm.auto import tqdm
from openai import OpenAI

from hyperopt import fmin, tpe, hp, STATUS_OK, Trials
from hyperopt.pyll import scope

/home/danny/Code/llm-zoomcamp-playground/.venv/lib/python3.11/site-packages/hyperopt/atpe.py:19: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


### Ingestion

In [2]:
df = pd.read_csv("data/data.csv")
documents = df.to_dict(orient="records")

In [3]:
documents[0]

{'id': 0,
 'exercise_name': 'Push-Ups',
 'type_of_activity': 'Strength',
 'type_of_equipment': 'Bodyweight',
 'body_part': 'Upper Body',
 'type': 'Push',
 'muscle_groups_activated': 'Pectorals, Triceps, Deltoids',
 'instructions': 'Start in a high plank position with your hands under your shoulders. Lower your body until your chest nearly touches the floor. Push back up to the starting position.'}

In [4]:
index = minsearch.AppendableIndex(
    text_fields=[
         'exercise_name',
         'type_of_activity',
         'type_of_equipment',
         'body_part',
         'type',
         'muscle_groups_activated',
         'instructions'
    ],
    keyword_fields=['id']
)

In [5]:
index.fit(documents)

### RAG

In [7]:
client = OpenAI()

zai_client = OpenAI(
    api_key=os.getenv('ZAI_API_KEY'),
    base_url='https://api.z.ai/api/paas/v4/'
)

groq_client = OpenAI(
    api_key=os.environ.get("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

In [8]:
def search(query):
    boost = {}
    results = index.search(
        query=query,
        filter_dict={},
        boost_dict=boost,
        num_results=5
    )
    return results

In [9]:
def build_prompt(query, search_results):
    prompt_template = """
You're an expert fitness instructor. Answer the QUESTION based on the CONTEXT from our exercises database.
Use only the facts from the CONTEXT when answering the QUESTION.

QUESTION: {question}

CONTEXT:
{context}
""".strip()

    context_template = """
exercise_name: {exercise_name}
type_of_activity: {type_of_activity}
type_of_equipment: {type_of_equipment}
body_part: {body_part}
type: {type}
muscle_groups_activated: {muscle_groups_activated}
instructions: {instructions}
"""

    context = ""
    for doc in search_results:
        context += context_template.format(**doc) + "\n\n"

    prompt = prompt_template.format(question=query, context=context)
    return prompt

In [74]:
def llm(prompt, model="gpt-5-nano"):
    if model == "gpt-5-nano":
        response = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}]
        )
    elif model == "glm-4.5-flash":
        response = zai_client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}]
        )
    elif model == "llama-3.1-8b-instant":
        response = groq_client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}]
        )
    else:
        raise "Error: unsupported model!"
        
    return response.choices[0].message.content

In [75]:
def rag(query, model="gpt-5-nano"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt, model=model)
    return answer

In [76]:
question = "Which exercise is the best to gain bicep muscles?"
answer = rag(question)
print(answer)

In the provided context, the best options for gaining biceps are:

- Bicep Curls (dumbbells): activats Biceps and Forearms; curl dumbbells from arms fully extended to shoulders.
- Cable Bicep Curl (cable machine): activates Biceps and Forearms; curl the handle toward shoulders.

Note: The Superman Exercise targets the core and posterior chain, not the biceps.


### Retrieval Evaluation

In [56]:
df_gt = pd.read_csv("data/ground_truth_data.csv")
ground_truth = df_gt.to_dict(orient="records")

In [57]:
ground_truth[0]

{'doc_id': 0,
 'question': 'What is the correct starting position for a push-up?'}

In [15]:
def hit_rate(relevance_total):
    cnt = 0
    for line in relevance_total:
        if True in line:
            cnt += 1
    return cnt / len(relevance_total)


def mrr(relevance_total):
    total_score = 0.0
    for line in relevance_total:
        for rank in range(len(line)):
            if line[rank] == True:
                total_score += 1 / (rank + 1)
    return total_score / len(relevance_total)

In [16]:
def minsearch_search(query, boost={}):
    results = index.search(
        query=query,
        filter_dict={},
        boost_dict=boost,
        num_results=10
    )
    return results

In [17]:
def evaluate(ground_truth, search_fn, params={}):
    relevance_total = []
    
    for q in tqdm(ground_truth):
        doc_id = q["doc_id"]
        results = search_fn(q, params)
        relevance = [doc_id == d["id"] for d in results]
        relevance_total.append(relevance)
        
    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total)
    }

In [18]:
evaluate(ground_truth, lambda q, p: minsearch_search(q["question"]))

  0%|          | 0/45 [00:00<?, ?it/s]

{'hit_rate': 0.9777777777777777, 'mrr': 0.6925925925925924}

#### Tuning minsearch boost params

In [19]:
def objective(params):
    results = evaluate(ground_truth, lambda q, p: minsearch_search(q["question"], params))
    loss = -results["mrr"]
    
    return {
        "results": results,
        "loss": loss,
        'status': STATUS_OK
    }

In [20]:
search_space = {
    'exercise_name': hp.uniform('exercise_name', 0.0, 3.0),
    'type_of_activity': hp.uniform('type_of_activity', 0.0, 3.0),
    'type_of_equipment': hp.uniform('type_of_equipment', 0.0, 3.0),
    'body_part': hp.uniform('body_part', 0.0, 3.0),
    'type': hp.uniform('type', 0.0, 3.0),
    'muscle_groups_activated': hp.uniform('muscle_groups_activated', 0.0, 3.0),
    'instructions': hp.uniform('instructions', 0.0, 3.0),
}

best = fmin(
    fn=objective,
    space=search_space,
    algo=tpe.suggest,
    max_evals=10,
    trials=Trials()
)

  0%|                                                                                                 | 0/10 [00:00<?, ?trial/s, best loss=?]

  0%|          | 0/45 [00:00<?, ?it/s]

 10%|███████                                                               | 1/10 [00:00<00:01,  6.34trial/s, best loss: -0.6333333333333332]

  0%|          | 0/45 [00:00<?, ?it/s]

 20%|██████████████                                                        | 2/10 [00:00<00:01,  6.61trial/s, best loss: -0.6759259259259258]

...

 80%|████████████████████████████████████████████████████████              | 8/10 [00:01<00:00,  6.03trial/s, best loss: -0.6925925925925924]

  0%|          | 0/45 [00:00<?, ?it/s]

 90%|███████████████████████████████████████████████████████████████       | 9/10 [00:01<00:00,  6.19trial/s, best loss: -0.6925925925925924]

  0%|          | 0/45 [00:00<?, ?it/s]

100%|█████████████████████████████████████████████████████████████████████| 10/10 [00:01<00:00,  6.46trial/s, best loss: -0.6925925925925924]


In [21]:
boost = {
     'body_part': 1.3617719252298475,
     'exercise_name': 2.747748020851861,
     'instructions': 1.6961000266977542,
     'muscle_groups_activated': 0.5307260562229762,
     'type': 2.290634544364637,
     'type_of_activity': 2.8558751878100637,
     'type_of_equipment': 2.651131478469881
}

evaluate(ground_truth, lambda q, p: minsearch_search(q["question"], p), boost)

  0%|          | 0/45 [00:00<?, ?it/s]

{'hit_rate': 0.9777777777777777, 'mrr': 0.7092592592592591}

In [22]:
def search(query):
    boost = {
     'body_part': 1.3617719252298475,
     'exercise_name': 2.747748020851861,
     'instructions': 1.6961000266977542,
     'muscle_groups_activated': 0.5307260562229762,
     'type': 2.290634544364637,
     'type_of_activity': 2.8558751878100637,
     'type_of_equipment': 2.651131478469881
    }
    
    results = index.search(
        query=query,
        filter_dict={},
        boost_dict=boost,
        num_results=5
    )
    return results

### RAG Evaluation

In [92]:
eval_prompt_template = """
You are an expert evaluator for a RAG system.
Your task is to analyze the relevance of the generated answer to the given question.
Based on the relevance of the generated answer, you will classify it
as "NON_RELEVANT", "PARTLY_RELEVANT", or "RELEVANT".

Here is the data for evaluation:

Question: {question}
Generated Answer: {answer_llm}

Please analyze the content and context of the generated answer in relation to the question
and provide your evaluation in parsable JSON. Don't use code blocks.

{{
  "Relevance": "NON_RELEVANT" | "PARTLY_RELEVANT" | "RELEVANT",
  "Explanation": "[Provide a brief explanation for your evaluation]"
}}
""".strip()

In [77]:
gpt_5_nano_gt = [{**d} for d in ground_truth[:10]]
llama_3_1_8b_instant_gt = [{**d} for d in ground_truth[:10]]

In [78]:
for rec in tqdm(gpt_5_nano_gt):
    question = rec["question"]
    answer_llm = rag(question)
    rec["answer_llm"] = answer_llm

  0%|          | 0/10 [00:00<?, ?it/s]

In [81]:
for rec in tqdm(llama_3_1_8b_instant_gt):
    question = rec["question"]
    answer_llm = rag(question, "llama-3.1-8b-instant")
    rec["answer_llm"] = answer_llm

  0%|          | 0/10 [00:00<?, ?it/s]

In [113]:
gpt_5_nano_gt[7]

{'doc_id': 1,
 'question': 'Which muscle groups are primarily targeted during squats?',
 'answer_llm': 'Quadriceps, glutes, and hamstrings.'}

In [114]:
llama_3_1_8b_instant_gt[7]

{'doc_id': 1,
 'question': 'Which muscle groups are primarily targeted during squats?',
 'answer_llm': 'Based on the provided context, the primary muscle groups targeted during squats are:\n\n1. Quadriceps\n2. Glutes\n3. Hamstrings'}

#### Evaluation: LLM-as-a-Judge
- Compare the two model: gpt-5-nana vs. llama-3.1-8b-instance
- Evaluate by glm-4.5-flash

In [86]:
gpt_5_nano_evals = []
llama_3_1_8b_instant_evals = []

In [87]:
for rec in tqdm(gpt_5_nano_gt[:10]):
    question = rec["question"]
    answer_llm = rec["answer_llm"]
    
    prompt = eval_prompt_template.format(
        question=question, 
        answer_llm=answer_llm
    )
    evaluation = llm(prompt, "glm-4.5-flash")
    evaluation = json.loads(evaluation)

    gpt_5_nano_evals.append((rec, answer_llm, evaluation))

  0%|          | 0/10 [00:00<?, ?it/s]

In [93]:
for rec in tqdm(llama_3_1_8b_instant_gt[:10]):
    question = rec["question"]
    answer_llm = rec["answer_llm"]
    
    prompt = eval_prompt_template.format(
        question=question, 
        answer_llm=answer_llm
    )
    evaluation = llm(prompt, "glm-4.5-flash")
    evaluation = json.loads(evaluation)

    llama_3_1_8b_instant_evals.append((rec, answer_llm, evaluation))

  0%|          | 0/10 [00:00<?, ?it/s]

In [109]:
gpt_5_nano_evals[9]

({'doc_id': 1,
  'question': 'What should I do if I feel discomfort while doing squats?',
  'answer_llm': 'The provided CONTEXT does not specify what to do if you feel discomfort during squats. It only describes how to perform squats: stand with feet shoulder-width apart, lower your body as if sitting back into a chair while keeping your chest up, then return to standing.'},
 'The provided CONTEXT does not specify what to do if you feel discomfort during squats. It only describes how to perform squats: stand with feet shoulder-width apart, lower your body as if sitting back into a chair while keeping your chest up, then return to standing.',
 {'Relevance': 'NON_RELEVANT',
  'Explanation': "The generated answer explicitly states that the context does not provide guidance on handling discomfort during squats. Instead, it only describes proper squat form, which doesn't address the specific question about what to do when experiencing discomfort. The answer fails to provide any actionable a

In [110]:
llama_3_1_8b_instant_evals[9]

({'doc_id': 1,
  'question': 'What should I do if I feel discomfort while doing squats?',
  'answer_llm': "If you feel discomfort while doing squats, I would recommend stretching to see if you can alleviate the issue. Based on the Context, we have a Hamstring Stretch exercise that targets the Hamstrings muscle group, which is activated during squats.\n\nYou can follow the instructions for the Hamstring Stretch exercise: Stand upright and place one heel on a bench or step, then lean forward slightly to feel a stretch in the hamstring of the elevated leg.\n\nAdditionally, if you're experiencing discomfort, you may want to check your form and make sure you're not putting too much strain on your lower body. Make sure to keep your chest up, engage your core, and lower your body as if sitting back into a chair.\n\nIf the discomfort persists, it may be worth consulting with a medical professional to rule out any potential injuries. However, in the meantime, stretching and adjusting your form 

In [99]:
df_gpt_5_nano_evals = pd.DataFrame(gpt_5_nano_evals, columns=["record", "answer", "evaluation"])

df_gpt_5_nano_evals["doc_id"] = df_gpt_5_nano_evals.record.apply(lambda d: d["doc_id"])
df_gpt_5_nano_evals["question"] = df_gpt_5_nano_evals.record.apply(lambda d: d["question"])
df_gpt_5_nano_evals["relevance"] = df_gpt_5_nano_evals.evaluation.apply(lambda d: d["Relevance"])
df_gpt_5_nano_evals["explanation"] = df_gpt_5_nano_evals.evaluation.apply(lambda d: d["Explanation"])

del df_gpt_5_nano_evals["record"]
del df_gpt_5_nano_evals["evaluation"]

In [102]:
df_llama_3_1_8b_instant_evals = pd.DataFrame(llama_3_1_8b_instant_evals, columns=["record", "answer", "evaluation"])

df_llama_3_1_8b_instant_evals["doc_id"] = df_llama_3_1_8b_instant_evals.record.apply(lambda d: d["doc_id"])
df_llama_3_1_8b_instant_evals["question"] = df_llama_3_1_8b_instant_evals.record.apply(lambda d: d["question"])
df_llama_3_1_8b_instant_evals["relevance"] = df_llama_3_1_8b_instant_evals.evaluation.apply(lambda d: d["Relevance"])
df_llama_3_1_8b_instant_evals["explanation"] = df_llama_3_1_8b_instant_evals.evaluation.apply(lambda d: d["Explanation"])

del df_llama_3_1_8b_instant_evals["record"]
del df_llama_3_1_8b_instant_evals["evaluation"]

In [104]:
df_gpt_5_nano_evals.relevance.value_counts()

relevance
RELEVANT           8
PARTLY_RELEVANT    1
NON_RELEVANT       1
Name: count, dtype: int64

In [105]:
df_llama_3_1_8b_instant_evals.relevance.value_counts()

relevance
RELEVANT    10
Name: count, dtype: int64

In [115]:
df_gpt_5_nano_evals.to_csv("evals/rag_eval_gpt_5_nano.csv")
df_llama_3_1_8b_instant_evals.to_csv("evals/rag_eval_llama_3_1_8b_instant.csv")